# 01 — Data Collection

Fetches 15 Wikipedia articles about the 2026 FIFA World Cup and related topics,
chunks them into retrievable segments, and saves the result to `data/chunks.json`.

**Run order:** This notebook must be run **first** before any retrieval notebooks.

**Output:** `../data/chunks.json` — list of chunk dicts with keys `id`, `source`, `url`, `text`.

**Reference:** EXPERIMENT.md §Dataset for target pages and chunking strategy.

In [ ]:
# Cell 1 — Install dependencies
# Install if not already present
!pip install wikipedia-api==0.6.0 tqdm

In [ ]:
# Cell 2 — Imports
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.chunker import fetch_wikipedia_page, build_chunk_list, save_chunks
from tqdm import tqdm

In [ ]:
# Cell 3 — Define target pages
# 15 Wikipedia articles as specified in EXPERIMENT.md §Dataset
TARGET_PAGES = [
    "2026 FIFA World Cup",
    "2026 FIFA World Cup squads",
    "FIFA World Cup records and statistics",
    "2022 FIFA World Cup",
    "Lionel Messi",
    "Kylian Mbappé",
    "Cristiano Ronaldo",
    "Erling Haaland",
    "Lamine Yamal",
    "Vinicius Jr.",
    "Jude Bellingham",
    "History of the FIFA World Cup",
    "Association football",
    "FIFA",
    "Goalkeeper (association football)"
]

print(f"Target pages: {len(TARGET_PAGES)}")
for i, p in enumerate(TARGET_PAGES, 1):
    print(f"  {i:2}. {p}")

In [ ]:
# Cell 4 — Fetch all pages with tqdm progress bar
pages = []
failed = []

for title in tqdm(TARGET_PAGES, desc="Fetching Wikipedia pages"):
    result = fetch_wikipedia_page(title)
    if result is not None:
        pages.append(result)
        print(f"  [OK]  {title} ({len(result['text'])} chars)")
    else:
        failed.append(title)
        print(f"  [FAIL] {title} — page not found")

print(f"\nFetched: {len(pages)} / {len(TARGET_PAGES)} pages")
if failed:
    print(f"Failed:  {failed}")

In [ ]:
# Cell 5 — Build chunk list and show stats
chunks = build_chunk_list(pages)

print(f"Total chunks: {len(chunks)}")
print()

# Chunks per source table
from collections import Counter
counts = Counter(c['source'] for c in chunks)
print(f"{'Source':<45} {'Chunks':>6}")
print("-" * 53)
for source, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"{source:<45} {n:>6}")

print()
print("Sample chunk (id=0):")
print(f"  Source: {chunks[0]['source']}")
print(f"  URL   : {chunks[0]['url']}")
print(f"  Text  : {chunks[0]['text'][:300]} ...")

In [ ]:
# Cell 6 — Save chunks to disk
save_chunks(chunks, "../data/chunks.json")